# Crater Signature Characterization — Australia

**The question (from Anna):** can gravity + magnetic give a *sufficiently unique*
signature to identify impact structures? Rather than detect craters blindly, we
**characterize the known ones** and test whether they form a separable pattern.

This is *not* the image-based deep learning of the Moon/Mars repos (YOLO/CNN on
optical photos). Buried Earth craters have no optical rim. We work in potential
fields, asking whether known craters cluster distinctly from random crust.

**Expected honest result:** a pattern exists for *large* craters (≳20 km) and
vanishes below the model resolution (~25 km). Quantifying that floor is itself the
finding, and it points the candidate search at large structures.

## 1. Imports + crater catalog (from your CSV)

In [1]:
import numpy as np, csv, io
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

CSV = """name,lat,lon,diameter_km,status
Acraman,-32.1,135.1,90,confirmed
Amelia Creek,-20.55,134.50,20,confirmed
Boxhole,-22.37,135.12,0.17,confirmed
Cleanskin,-18.10,137.56,15,unconfirmed
Connolly Basin,-23.32,124.45,9,confirmed
Crawford,-34.43,139.2,8.5,confirmed
Dalgaranga,-27.38,117.17,0.02,confirmed
Flaxman,-34.37,139.4,10,confirmed
Foelsche,-15.40,136.47,6,confirmed
Glikson,-23.59,121.34,19,confirmed
Goat Paddock,-18.20,126.40,5.1,confirmed
Gosses Bluff,-23.49,132.19,22,confirmed
Goyder,-13.28,135.2,3,confirmed
Henbury,-24.34,133.8,0.15,confirmed
Hickman,-23.02,119.41,0.26,confirmed
Kelly West,-19.56,133.57,10,confirmed
Lawn Hill,-18.40,138.39,18,confirmed
Liverpool,-12.24,134.3,1.6,confirmed
Matt Wilson,-15.30,131.11,7.5,confirmed
Mount Toondina,-27.57,135.22,4,confirmed
Piccaninny,-17.32,128.25,7,confirmed
Shoemaker,-25.52,120.53,30,confirmed
Spider,-16.44,126.5,13,confirmed
Strangways,-15.12,133.35,25,confirmed
Tookoonooka,-27.7,142.50,55,confirmed
Veevers,-22.58,125.22,0.08,confirmed
Wolfe Creek,-19.10,127.48,0.87,confirmed
Woodleigh,-26.3,114.39,40,confirmed
Yallalie,-30.28,115.47,15,unconfirmed
Yarrabubba,-27.10,118.5,30,confirmed
"""

def load_craters():
    out=[]
    for r in csv.DictReader(io.StringIO(CSV)):
        # NB: Australia is southern hemisphere -> force negative latitude
        out.append((r["name"], -abs(float(r["lat"])), float(r["lon"]),
                    float(r["diameter_km"]), r["status"]))
    return out

craters = load_craters()
print(len(craters), "craters |",
      sum(c[3]>=20 for c in craters), "are >=20 km (resolvable at SH resolution)")

30 craters | 8 are >=20 km (resolvable at SH resolution)


## 2. Data layer — SWAP for real pyshtools fetches when running locally

`get_field_patch` returns an (n×n) patch around a location. Replace the synthetic
body with: synthesize XGM2019E (gravity) and NGDC_720_V3 (magnetic) globally once
(see the EoS/gravity notebook for the `expand(... normal_gravity=True, a, f, omega)`
pattern), then slice the window for each (lat,lon). Everything downstream is unchanged.

Here it plants crater-like signatures at the real coordinates so the logic is testable
offline. Note both models are ~25 km resolution, so small craters are sub-pixel — by
design, so the resolution finding is faithfully reproduced.

In [2]:
AUS = dict(lat0=-39, lat1=-11, lon0=113, lon1=154)

def get_field_patch(lat, lon, kind="gravity", n=24, _synth=None):
    rng = np.random.default_rng(int(abs(lat*1000+lon*7)) % (2**32))
    yy, xx = np.mgrid[0:n, 0:n]; c=n/2.0
    field = 3*np.sin(xx/7.0+lat) + 2*np.cos(yy/6.0+lon/10) + rng.normal(0,0.6,(n,n))
    if _synth is not None:
        for clat,clon,D in _synth:
            if abs(clat-lat)<1e-4 and abs(clon-lon)<1e-4:
                rr=np.sqrt((xx-c)**2+(yy-c)**2); r_px=max((D/2)/9.0,0.8)
                amp = (0.04*D+0.8) if kind=="magnetic" else (-0.05*D-1.0)
                field += amp*np.exp(-(rr**2)/(2*(r_px*0.7)**2))
    return field

def feat_vector(lat, lon, _synth=None):
    g=get_field_patch(lat,lon,"gravity",_synth=_synth)
    m=get_field_patch(lat,lon,"magnetic",_synth=_synth)
    n=g.shape[0]; c=n//2; yy,xx=np.mgrid[0:n,0:n]; rr=np.sqrt((xx-c)**2+(yy-c)**2)
    inner=rr<n*0.18; outer=(rr>n*0.30)&(rr<n*0.45); mid=(rr>n*0.2)&(rr<n*0.35)
    def f(a):
        contrast=a[inner].mean()-a[outer].mean()
        sym=1-a[mid].std()/(abs(a[mid].mean())+abs(contrast)+1e-6)
        return contrast, float(np.clip(sym,0,1)), float(np.ptp(a[rr<n*0.45]))
    return [*f(g), *f(m)]

FEAT_NAMES=["grav_contrast","grav_sym","grav_relief",
            "mag_contrast","mag_sym","mag_relief"]
synth=[(c[1],c[2],c[3]) for c in craters]
print("feature vector at Acraman:", [round(v,2) for v in feat_vector(craters[0][1],craters[0][2],synth)])

feature vector at Acraman: [-3.48, 0.59, 9.81, 2.93, 0.64, 9.5]


## 3. Build feature matrices — craters vs random controls

In [3]:
def controls(n=200, seed=7):
    rng=np.random.default_rng(seed); out=[]
    while len(out)<n:
        la=rng.uniform(AUS["lat0"],AUS["lat1"]); lo=rng.uniform(AUS["lon0"],AUS["lon1"])
        if all(np.hypot(la-c[1],lo-c[2])>0.5 for c in craters): out.append((la,lo))
    return np.array([feat_vector(la,lo,synth) for la,lo in out])

Xk = controls()
Xc_all = np.array([feat_vector(lat,lon,synth) for _,lat,lon,_,_ in craters])
print("controls:", Xk.shape, "| craters:", Xc_all.shape)

controls: (200, 6) | craters: (30, 6)


## 4. The key test — does the pattern sharpen for large craters?

This is the scientific result. If AUC for *all* craters is ~chance but AUC for large
craters is meaningfully higher, we've quantified that gravity+magnetic uniquely
characterize only large impact structures at this resolution.

In [4]:
for label, dmin in [("ALL craters",0), ("D>=20km (resolvable)",20)]:
    sel=[c for c in craters if c[3]>=dmin]
    Xc=np.array([feat_vector(lat,lon,synth) for _,lat,lon,_,_ in sel])
    X=np.vstack([Xc,Xk]); y=np.r_[np.ones(len(Xc)),np.zeros(len(Xk))]
    rf=RandomForestClassifier(n_estimators=200,max_depth=4,
                              class_weight="balanced",random_state=0)
    auc=cross_val_score(rf,X,y,cv=4,scoring="roc_auc")
    print(f"{label:24s} n_pos={len(sel):2d}  ROC-AUC = {auc.mean():.3f} +/- {auc.std():.3f}")

ALL craters              n_pos=30  ROC-AUC = 0.493 +/- 0.055
D>=20km (resolvable)     n_pos= 8  ROC-AUC = 0.698 +/- 0.130


## 5. From pattern to candidates (the field-trip path)

Once the large-crater pattern is established on **real** data, train the classifier on
the resolvable known craters, then score a grid of locations across Australia. Rank by
crater-similarity probability, drop anything within ~0.5 deg of a known crater, and the
top remaining locations are your candidates. Cross-check each in QGIS against geology
(rule out plutons/volcanics/salt) and the Earth Impact Database (confirm not catalogued).
A candidate that is circular in both gravity and magnetic, unexplained geologically, and
uncatalogued is what justifies a field trip.

**Honest caveat to carry into the writeup:** with only ~8 resolvable positives, the
classifier is statistically thin and the candidate list is suggestive, not conclusive.
That is exactly the right framing for a desk study proposing field follow-up.

In [5]:
# Skeleton for the real-data candidate scan (runs once real fetches are wired in):
def scan_candidates(step_deg=0.5, dmin=20, top=10):
    big=[c for c in craters if c[3]>=dmin]
    Xc=np.array([feat_vector(lat,lon,synth) for _,lat,lon,_,_ in big])
    X=np.vstack([Xc,Xk]); y=np.r_[np.ones(len(Xc)),np.zeros(len(Xk))]
    rf=RandomForestClassifier(n_estimators=300,max_depth=4,
                              class_weight="balanced",random_state=0).fit(X,y)
    grid=[]
    for la in np.arange(AUS["lat0"],AUS["lat1"],step_deg):
        for lo in np.arange(AUS["lon0"],AUS["lon1"],step_deg):
            if all(np.hypot(la-c[1],lo-c[2])>0.5 for c in craters):
                p=rf.predict_proba([feat_vector(la,lo,synth)])[0,1]
                grid.append((p,la,lo))
    grid.sort(reverse=True)
    return grid[:top]

print("Top synthetic candidate cells (prob, lat, lon):")
for p,la,lo in scan_candidates():
    print(f"  p={p:.3f}  ({la:.1f}, {lo:.1f})")

Top synthetic candidate cells (prob, lat, lon):
  p=0.518  (-13.5, 114.0)
  p=0.518  (-31.0, 118.0)
  p=0.480  (-38.5, 115.0)
  p=0.443  (-38.0, 120.0)
  p=0.429  (-27.5, 130.0)
  p=0.423  (-27.5, 131.0)
  p=0.422  (-16.5, 149.0)
  p=0.419  (-31.0, 116.5)
  p=0.419  (-35.0, 118.0)
  p=0.418  (-18.5, 116.5)
